# Listado de Variables para el Modelo de Recomendación y Análisis Turístico

Este conjunto de datos consolida variables de **demanda, oferta, percepción y contexto** para nutrir el motor de recomendación y el análisis de atribución de mercado (TUI Project).

---

## 1. Variables Climáticas y Ambientales
*Factores condicionantes de la estacionalidad y la satisfacción directa del viajero.*

* **Temperatura Media Mensual (°C):** Máximas, mínimas y promedios para detectar ventanas óptimas de viaje.
* **Precipitación Acumulada y Días de Lluvia:** Número de días con lluvia al mes (métrica más representativa del impacto en la estancia que los mm totales).
* **Horas de Sol Efectivas / Índice UV:** Indicador clave para destinos de producto Sol y Playa.
* **Temperatura del Agua del Mar (°C):** Relevante para destinos de costa y deportes acuáticos.
* **Índice de Confort Térmico (Sensación Térmica / Humedad):** Medición de la combinación entre temperatura y humedad relativa (clave en zonas tropicales o mediterráneas en verano).

---

## 2. Accesibilidad y Conectividad en Transporte
*Determinantes de la facilidad de acceso y el volumen potencial de flujo turístico desde el mercado emisor.*

* **Vuelos Directos y Frecuencias Semanales:** Conexiones directas operativas desde los hubs emisores (UK, DE, ES).
* **Capacidad de Asientos Programados (*Seat Capacity*):** Asientos totales ofertados por aerolíneas hacia el destino.
* **Presencia de Aerolíneas Low Cost (LCC):** Indicador de accesibilidad económica y atracción de turismo de escapada/joven.
* **Tiempo de Viaje y Escalas:** Duración media total del trayecto desde la capital o aeropuerto emisor de referencia.

---

## 3. Indicadores Económicos y Coste en Destino
*Variables financieras que condicionan la elección en función del presupuesto del usuario.*

* **Precio Medio Diario por Habitación (*ADR / RevPAR*):** Evolución temporal del coste de alojamiento.
* **Índice de Coste de Vida Turístico:** 
  * Precio medio de la oferta de restauración (menú diario / cena).
  * Precio medio de la cesta básica vacacional (bebidas, ocio, transporte local).
* **Tipo de Cambio y Paridad de Divisa:** Variación cambiaria frente al Euro (EUR) o Libra (GBP) para destinos fuera de la Eurozona (ej. Georgia, Tailandia, Vietnam, Turquía).

---

## 4. Seguridad, Infraestructura y Salud
*Indicadores de fricción y factores de decisión en la fase de consideración.*

* **Índice de Seguridad / Tasa de Criminalidad Percibida:** Evaluación de la percepción de seguridad en el destino.
* **Calidad de la Infraestructura Sanitaria:** Accesibilidad a centros médicos y cobertura hospitalaria para turistas.
* **Requisitos Burocráticos y Visados:** 
  * Pertenencia a espacio Schengen.
  * Exención de visado / Visado a la llegada (*Visa on Arrival*) / Trámites previos (e-Visa).

---

## 5. Estacionalidad, Eventos y Calendario Emisor
*Variables temporales que explican picos puntuales de interés en motores de búsqueda (Google Trends).*

* **Vacaciones Escolares y Festivos en Mercados Emisores:** Calendario de *Half-Terms* (UK), *Pfingsten/Sommerferien* (DE) y festivos nacionales (ES).
* **Grandes Eventos y Festivales:** Calendario de eventos deportivos, musicales o culturales con impacto en la ocupación y precios.

---

## 6. Atractivos y Tipología del Destino (Categorización y Clustering)
*Atributos estructurales utilizados para la recomendación por similitud entre destinos.*

* **Kilómetros de Playa y Banderas Azules:** Extensión de costa y sellos de calidad ambiental.
* **Patrimonio Cultural / UNESCO:** Densidad de sitios declarados Patrimonio de la Humanidad o puntos de interés cultural.
* **Perfil de Oferta Complementaria:** 
  * Densidad de oferta gastronómica (locales tradicionales vs. alta cocina).
  * Opciones de ocio nocturno vs. turismo activo / naturaleza.

---

## Arquitectura de Datos Sugerida

In [ ]:
import os
import time
import pandas as pd
import requests
from geopy.geocoders import Nominatim

# ==============================================================================
# 0. DEFINICIÓN DE LA RUTA DE GUARDADO
# ==============================================================================
RUTA_DESCARGAS = r"C:\Users\mtkyg\Downloads"
NOMBRE_ARCHIVO = "clima_todos_los_destinos.csv"
RUTA_COMPLETA_SALIDA = os.path.join(RUTA_DESCARGAS, NOMBRE_ARCHIVO)

# ==============================================================================
# 1. LISTAS ORIGINALES DE DESTINOS Y GRUPOS
# ==============================================================================
DESTINOS_CIUDADES_ESPANOLAS = [
    'Alicante', 'Barcelona', 'Benalmádena', 'Benidorm', 'Bilbao', 'Cartagena', 
    'Chiclana de la Frontera', 'Coruña', 'Cádiz', 'Córdoba', 'San Sebastian', 
    'Eivissa', 'Elche', 'Fuengirola', 'Gijón', 'Granada', 'Jerez de la Frontera', 
    'Madrid', 'Marbella', 'Murcia', 'Málaga', 'Oviedo', 'Palma', 'Palmas de Gran Canaria', 
    'Pamplona', 'Puerto de Santa María', 'Puerto de la Cruz', 'Salamanca', 
    'Santa Cruz de Tenerife', 'Santiago de Compostela', 'Sevilla', 'Toledo', 
    'Torremolinos', 'Torrevieja', 'Valladolid', 'València', 'Vigo', 'Vitoria', 'Zaragoza'
]

DESTINOS_GRUPOS = [
    # Grupo 1 - Emergentes Mediterráneo seguro
    ["Georgia Batumi", "Eslovenia Portoroz", "Croacia islas", "Grecia Lefkada", "Grecia Thassos", "Italia Tropea"],
    # Grupo 2 - Asia segura
    ["Tailandia Phuket", "Tailandia Koh Samui", "Bali", "Vietnam Da Nang", "Sri Lanka", "Maldivas"],
    # Grupo 3 - Océano Índico
    ["Mauricio", "Seychelles", "Reunión", "Zanzíbar", "Maldivas atolón"],
    # Grupo 4 - Caribe ampliado seguro
    ["Antigua", "Santa Lucía", "Granada Caribe", "Dominica", "Bonaire", "Turks and Caicos"],
    # Grupo 5 - Naturaleza seguros
    ["Islandia", "Noruega fiordos", "Azores senderismo", "Madeira trekking", "Canarias volcanes", "Escocia Highlands"],
    # Grupo 6 - Oriente Medio seguro
    ["Dubái", "Abu Dhabi", "Omán Muscat", "Omán Salalah", "Jordania Mar Muerto", "Jordania Aqaba"],
    # Grupo 7 - México y Centroamérica
    ["Playa del Carmen", "Tulum", "Los Cabos", "Puerto Vallarta", "Costa Rica", "Panamá"],
    # Grupo 8 - Cabo Verde
    ["Cabo Verde Sal", "Cabo Verde Boa Vista", "Cabo Verde Santiago", "Cabo Verde Santo Antão"],
    # Grupo 9 - Marruecos
    ["Marrakech", "Agadir", "Essaouira", "Fez", "Tánger"],
    # Grupo 10 - Italia ampliada
    ["Sicilia", "Cerdeña", "Calabria", "Puglia", "Costa Amalfitana", "Toscana", "Cinque Terre", "Lago de Garda"],
    # Grupo 11 - Croacia y Adriático
    ["Dubrovnik", "Split", "Istria", "Hvar", "Montenegro Budva", "Albania Saranda", "Albania Ksamil"],
    # Grupo 12 - Turquía ampliada
    ["Antalya", "Bodrum", "Dalaman", "Fethiye", "Cesme", "Kusadasi", "Marmaris", "Alanya"],
    # Grupo 13 - Egipto ampliado
    ["Hurghada", "Sharm el Sheikh", "Marsa Alam", "El Gouna", "Luxor"],
    # Grupo 14 - Grecia continental
    ["Atenas", "Salónica", "Pelión", "Halkidiki", "Peloponeso"],
    # Grupo 15 - Europa otros seguros
    ["Malta", "Chipre", "Chipre Norte", "Eslovenia costa", "Bulgaria Sunny Beach"],
    # Grupo 16 - Portugal atlántico
    ["Algarve", "Madeira", "Azores", "Lisboa costa", "Porto"],
    # Grupo 17 - Temporadas y ofertas
    ["vacaciones verano 2025 oferta", "viaje invierno sol barato", "Semana Santa playa", "Navidad Caribe", "puente mayo destino", "última hora vacaciones playa"],
    # Grupo 18 - Comparativas
    ["Mallorca vs Creta", "Cancún vs Punta Cana", "Tenerife vs Gran Canaria", "Antalya vs Hurghada", "Maldivas vs Mauricio", "Costa del Sol vs Algarve", "Bali vs Tailandia"],
    # Grupo 19 - Temáticas
    ["vacaciones playa familia Europa", "mejor destino luna de miel", "todo incluido barato", "mejor isla griega parejas", "destino sostenible Europa", "viaje aventura seguro"],
    # Grupo 20 - Marcas TUI
    ["TUI Blue Mallorca", "TUI Magic Life Bodrum", "TUI Sensatori Tenerife", "Robinson Club Fuerteventura", "RIU Cancún", "Iberostar Creta", "Barceló Bávaro"],
    # Grupo 21 - Baleares
    ["Mallorca", "Ibiza", "Menorca", "Formentera"],
    # Grupo 22 - Canarias
    ["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera"],
    # Grupo 23 - Costas España
    ["Costa del Sol", "Costa Brava", "Costa Blanca", "Costa Dorada", "Almería", "Cádiz", "Huelva"],
    # Grupo 24 - Grecia islas
    ["Creta", "Santorini", "Rodas", "Kos", "Corfu", "Zante", "Mykonos", "Kefalonia", "Skiathos", "Paros", "Naxos"],
    # Grupo 25 - Caribe clásico
    ["Cancún", "Riviera Maya", "Punta Cana", "Cuba Varadero", "Jamaica Montego Bay", "Aruba", "Curaçao", "Barbados"],
    # Grupo 26 - Opiniones español
    ["Mallorca vacaciones opiniones", "Tenerife hotel todo incluido", "Cancún experiencia viaje", "Creta playa familiar", "Punta Cana resort opiniones", "Antalya all inclusive"],
    # Grupo 27 - Reviews inglés
    ["Mallorca holiday review", "Tenerife all inclusive", "Cancun resort review", "Crete family holiday", "Rhodes travel tips", "Maldives overwater villa review"],
    # Grupo 28 - Bewertungen alemán
    ["Mallorca Urlaub Erfahrung", "Teneriffa Hotel Bewertung", "Kreta Familienurlaub", "Antalya All Inclusive", "Hurghada Reise Tipps", "Fuerteventura Strand"],
]

# ==============================================================================
# 2. DEPURACIÓN Y NORMALIZACIÓN DE LUGARES EN ESPAÑOL
# ==============================================================================
MAPEO_NORMALIZACION = {
    "Mallorca Urlaub Erfahrung": "Mallorca",
    "Teneriffa Hotel Bewertung": "Tenerife",
    "Kreta Familienurlaub": "Creta",
    "Antalya All Inclusive": "Antalya",
    "Hurghada Reise Tipps": "Hurghada",
    "Fuerteventura Strand": "Fuerteventura",
    "Mallorca holiday review": "Mallorca",
    "Tenerife all inclusive": "Tenerife",
    "Cancun resort review": "Cancún",
    "Crete family holiday": "Creta",
    "Rhodes travel tips": "Rodas",
    "Maldives overwater villa review": "Maldivas",
    "Mallorca vacaciones opiniones": "Mallorca",
    "Tenerife hotel todo incluido": "Tenerife",
    "Cancún experiencia viaje": "Cancún",
    "Creta playa familiar": "Creta",
    "Punta Cana resort opiniones": "Punta Cana",
    "TUI Blue Mallorca": "Mallorca",
    "TUI Magic Life Bodrum": "Bodrum",
    "TUI Sensatori Tenerife": "Tenerife",
    "Robinson Club Fuerteventura": "Fuerteventura",
    "RIU Cancún": "Cancún",
    "Iberostar Creta": "Creta",
    "Barceló Bávaro": "Punta Cana",
    "Georgia Batumi": "Batumi, Georgia",
    "Eslovenia Portoroz": "Portoroz, Eslovenia",
    "Croacia islas": "Hvar, Croacia",
    "Grecia Lefkada": "Lefkada, Grecia",
    "Grecia Thassos": "Thassos, Grecia",
    "Italia Tropea": "Tropea, Italia",
    "Tailandia Phuket": "Phuket, Tailandia",
    "Tailandia Koh Samui": "Koh Samui, Tailandia",
    "Vietnam Da Nang": "Da Nang, Vietnam",
    "Maldivas atolón": "Malé, Maldivas",
    "Granada Caribe": "Saint George's, Granada",
    "Noruega fiordos": "Bergen, Noruega",
    "Azores senderismo": "Ponta Delgada, Azores",
    "Madeira trekking": "Funchal, Madeira",
    "Canarias volcanes": "Lanzarote, España",
    "Escocia Highlands": "Inverness, Escocia",
    "Omán Muscat": "Mascate, Omán",
    "Omán Salalah": "Salalah, Omán",
    "Jordania Mar Muerto": "Mar Muerto, Jordania",
    "Jordania Aqaba": "Áqaba, Jordania",
    "Cabo Verde Sal": "Isla de Sal, Cabo Verde",
    "Cabo Verde Boa Vista": "Boa Vista, Cabo Verde",
    "Cabo Verde Santiago": "Santiago, Cabo Verde",
    "Cabo Verde Santo Antão": "Santo Antão, Cabo Verde",
    "Montenegro Budva": "Budva, Montenegro",
    "Albania Saranda": "Sarandë, Albania",
    "Albania Ksamil": "Ksamil, Albania",
    "Eslovenia costa": "Koper, Eslovenia",
    "Bulgaria Sunny Beach": "Sunny Beach, Bulgaria",
    "Lisboa costa": "Cascais, Portugal",
    "Cuba Varadero": "Varadero, Cuba",
    "Jamaica Montego Bay": "Montego Bay, Jamaica"
}

lugares_set = set()

for c in DESTINOS_CIUDADES_ESPANOLAS:
    lugares_set.add(c)

for grupo in DESTINOS_GRUPOS:
    for elem in grupo:
        if any(kw in elem for kw in ["vs", "oferta", "barato", "luna de miel", "sostenible", "familia", "Semana Santa", "Navidad", "puente mayo", "última hora", "aventura"]):
            continue
        lugar_clean = MAPEO_NORMALIZACION.get(elem, elem)
        lugares_set.add(lugar_clean)

LISTA_LUGARES_FINAL = sorted(list(lugares_set))
print(f"Total de lugares limpios y unificados a procesar: {len(LISTA_LUGARES_FINAL)}")

# ==============================================================================
# 3. OBTENCIÓN AUTOMÁTICA DE COORDENADAS (GEOCODIFICACIÓN CON GEOPY)
# ==============================================================================
geolocator = Nominatim(user_agent="meteo_ucm_research")
destinos_geocodificados = []

print("\nGeocodificando destinos (obteniendo Latitud y Longitud)...")

for lugar in LISTA_LUGARES_FINAL:
    try:
        query = lugar if ("España" in lugar or "," in lugar) else f"{lugar}, España" if lugar in DESTINOS_CIUDADES_ESPANOLAS else lugar
        location = geolocator.geocode(query, timeout=10)
        
        if location:
            destinos_geocodificados.append({
                "lugar": lugar,
                "lat": round(location.latitude, 4),
                "lon": round(location.longitude, 4)
            })
            print(f"  ✓ {lugar} -> Lat: {round(location.latitude, 4)}, Lon: {round(location.longitude, 4)}")
        else:
            print(f"  ✗ No encontrado directamente: {lugar}")
        time.sleep(1)
    except Exception as e:
        print(f"  [!] Error con {lugar}: {e}")

df_coordenadas = pd.DataFrame(destinos_geocodificados)

# ==============================================================================
# 4. EXTRACCIÓN DE DATOS METEOROLÓGICOS Y MARÍTIMOS CON OPEN-METEO
# ==============================================================================
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
UMBRAL_LLUVIA = 1.0

list_df_mensual = []

print("\nDescargando datos climáticos desde Open-Meteo...")

for idx, row in df_coordenadas.iterrows():
    lugar = row["lugar"]
    lat = row["lat"]
    lon = row["lon"]
    print(f" -> Procesando clima [{idx+1}/{len(df_coordenadas)}]: {lugar}")
    
    # 1. API Clima Atmosférico
    url_clima = "https://archive-api.open-meteo.com/v1/archive"
    params_clima = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ["temperature_2m_mean", "precipitation_sum", "sunshine_duration"],
        "hourly": ["relative_humidity_2m"],
        "timezone": "auto"
    }
    
    try:
        res_clima = requests.get(url_clima, params=params_clima).json()
        
        df_daily = pd.DataFrame({
            "date_str": res_clima["daily"]["time"],
            "temp_mean": res_clima["daily"]["temperature_2m_mean"],
            "precip_sum": res_clima["daily"]["precipitation_sum"],
            "horas_sol": [s / 3600.0 if s is not None else 0 for s in res_clima["daily"]["sunshine_duration"]]
        })
        
        df_hourly = pd.DataFrame({
            "date_str": [t[:10] for t in res_clima["hourly"]["time"]],
            "humedad": res_clima["hourly"]["relative_humidity_2m"]
        })
        df_hum = df_hourly.groupby("date_str", as_index=False)["humedad"].mean()
        df_clima = pd.merge(df_daily, df_hum, on="date_str", how="left")
    except Exception as e:
        print(f"    [!] Error al descargar clima de {lugar}: {e}")
        continue

    # 2. API Clima Marítimo
    url_marine = "https://marine-api.open-meteo.com/v1/marine"
    params_marine = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ["sea_surface_temperature"],
        "timezone": "auto"
    }
    try:
        res_marine = requests.get(url_marine, params=params_marine).json()
        if "hourly" in res_marine and "sea_surface_temperature" in res_marine["hourly"]:
            df_hourly_mar = pd.DataFrame({
                "date_str": [t[:10] for t in res_marine["hourly"]["time"]],
                "temp_agua": res_marine["hourly"]["sea_surface_temperature"]
            })
            df_marine = df_hourly_mar.groupby("date_str", as_index=False)["temp_agua"].mean()
        else:
            df_marine = pd.DataFrame(columns=["date_str", "temp_agua"])
    except Exception:
        df_marine = pd.DataFrame(columns=["date_str", "temp_agua"])

    # 3. Integración y Agregación Mensual
    df_full = pd.merge(df_clima, df_marine, on="date_str", how="left")
    df_full["lugar"] = lugar
    df_full["year_month"] = df_full["date_str"].str.slice(0, 7)
    
    df_monthly = df_full.groupby(["lugar", "year_month"], as_index=False).agg(
        temp_media_aire_c=("temp_mean", "mean"),
        temp_media_agua_c=("temp_agua", "mean"),
        precipitacion_total_mm=("precip_sum", "sum"),
        dias_lluvia=("precip_sum", lambda x: (x >= UMBRAL_LLUVIA).sum()),
        horas_sol_totales=("horas_sol", "sum"),
        humedad_media_pct=("humedad", "mean")
    )
    list_df_mensual.append(df_monthly)

# ==============================================================================
# 5. CONSOLIDACIÓN Y EXPORTACIÓN A DOWNLOADS
# ==============================================================================
if list_df_mensual:
    df_consolidado = pd.concat(list_df_mensual, ignore_index=True)
    
    # Redondeos
    df_consolidado["temp_media_aire_c"] = df_consolidado["temp_media_aire_c"].round(1)
    df_consolidado["temp_media_agua_c"] = df_consolidado["temp_media_agua_c"].round(1)
    df_consolidado["precipitacion_total_mm"] = df_consolidado["precipitacion_total_mm"].round(1)
    df_consolidado["horas_sol_totales"] = df_consolidado["horas_sol_totales"].round(1)
    df_consolidado["humedad_media_pct"] = df_consolidado["humedad_media_pct"].round(1)

    # Crear la carpeta de descargas si no existiese (por seguridad)
    os.makedirs(RUTA_DESCARGAS, exist_ok=True)

    # Guardar archivo CSV
    df_consolidado.to_csv(RUTA_COMPLETA_SALIDA, index=False)
    print(f"\n¡Proceso finalizado con éxito!")
    print(f"Archivo guardado directamente en: {RUTA_COMPLETA_SALIDA}")
    print("\n--- Muestra del Dataset ---")
    print(df_consolidado.head(15))

Total de lugares limpios y unificados a procesar: 169

Geocodificando destinos (obteniendo Latitud y Longitud)...
  ✓ Abu Dhabi -> Lat: 24.4538, Lon: 54.3774
  ✓ Agadir -> Lat: 30.4205, Lon: -9.5839
  ✓ Alanya -> Lat: 36.8866, Lon: 30.703
  ✓ Algarve -> Lat: 37.2449, Lon: -8.196
  ✓ Alicante -> Lat: 38.3436, Lon: -0.4882
  ✓ Almería -> Lat: 36.8686, Lon: -2.312
  ✓ Antalya -> Lat: 36.8866, Lon: 30.703
  ✓ Antalya all inclusive -> Lat: 36.6359, Lon: 31.7757
  ✓ Antigua -> Lat: 17.1037, Lon: -61.7905
  ✓ Aruba -> Lat: 12.5014, Lon: -69.9618
  ✓ Atenas -> Lat: 37.9756, Lon: 23.7348
  ✓ Azores -> Lat: 37.8086, Lon: -25.4731
  ✓ Bali -> Lat: -8.2271, Lon: 115.1919
  ✓ Barbados -> Lat: 13.15, Lon: -59.525
  ✓ Barcelona -> Lat: 41.3826, Lon: 2.1771
  ✓ Batumi, Georgia -> Lat: 41.651, Lon: 41.636
  ✓ Benalmádena -> Lat: 36.5945, Lon: -4.5723
  ✓ Benidorm -> Lat: 38.5406, Lon: -0.1291
  ✓ Bergen, Noruega -> Lat: 60.3943, Lon: 5.3259
  ✓ Bilbao -> Lat: 43.263, Lon: -2.935
  ✓ Boa Vista, Cabo Ver

## 2. Accesibilidad y Conectividad en Transporte
*Determinantes de la facilidad de acceso y el volumen potencial de flujo turístico desde el mercado emisor.*

* **Vuelos Directos y Frecuencias Semanales:** Conexiones directas operativas desde los hubs emisores (UK, DE, ES).
* **Capacidad de Asientos Programados (*Seat Capacity*):** Asientos totales ofertados por aerolíneas hacia el destino.
* **Presencia de Aerolíneas Low Cost (LCC):** Indicador de accesibilidad económica y atracción de turismo de escapada/joven.
* **Tiempo de Viaje y Escalas:** Duración media total del trayecto desde la capital o aeropuerto emisor de referencia.

In [15]:
import os
import pandas as pd
import numpy as np
import airportsdata

# ==============================================================================
# 0. TUS FUENTES DE DATOS EXACTAS
# ==============================================================================
DESTINOS_CIUDADES_ESPANOLAS = [
    'Alicante', 'Barcelona', 'Benalmádena', 'Benidorm', 'Bilbao', 'Cartagena', 
    'Chiclana de la Frontera', 'Coruña', 'Cádiz', 'Córdoba', 'San Sebastian', 
    'Eivissa', 'Elche', 'Fuengirola', 'Gijón', 'Granada', 'Jerez de la Frontera', 
    'Madrid', 'Marbella', 'Murcia', 'Málaga', 'Oviedo', 'Palma', 'Palmas de Gran Canaria', 
    'Pamplona', 'Puerto de Santa María', 'Puerto de la Cruz', 'Salamanca', 
    'Santa Cruz de Tenerife', 'Santiago de Compostela', 'Sevilla', 'Toledo', 
    'Torremolinos', 'Torrevieja', 'Valladolid', 'València', 'Vigo', 'Vitoria', 'Zaragoza'
]

DESTINOS_GRUPOS = [
    # Grupo 1 - Emergentes Mediterráneo seguro
    ["Georgia Batumi", "Eslovenia Portoroz", "Croacia islas", "Grecia Lefkada", "Grecia Thassos", "Italia Tropea"],
    # Grupo 2 - Asia segura
    ["Tailandia Phuket", "Tailandia Koh Samui", "Bali", "Vietnam Da Nang", "Sri Lanka", "Maldivas"],
    # Grupo 3 - Océano Índico
    ["Mauricio", "Seychelles", "Reunión", "Zanzíbar", "Maldivas atolón"],
    # Grupo 4 - Caribe ampliado seguro
    ["Antigua", "Santa Lucía", "Granada Caribe", "Dominica", "Bonaire", "Turks and Caicos"],
    # Grupo 5 - Naturaleza seguros
    ["Islandia", "Noruega fiordos", "Azores senderismo", "Madeira trekking", "Canarias volcanes", "Escocia Highlands"],
    # Grupo 6 - Oriente Medio seguro
    ["Dubái", "Abu Dhabi", "Omán Muscat", "Omán Salalah", "Jordania Mar Muerto", "Jordania Aqaba"],
    # Grupo 7 - México y Centroamérica
    ["Playa del Carmen", "Tulum", "Los Cabos", "Puerto Vallarta", "Costa Rica", "Panamá"],
    # Grupo 8 - Cabo Verde
    ["Cabo Verde Sal", "Cabo Verde Boa Vista", "Cabo Verde Santiago", "Cabo Verde Santo Antão"],
    # Grupo 9 - Marruecos
    ["Marrakech", "Agadir", "Essaouira", "Fez", "Tánger"],
    # Grupo 10 - Italia ampliada
    ["Sicilia", "Cerdeña", "Calabria", "Puglia", "Costa Amalfitana", "Toscana", "Cinque Terre", "Lago de Garda"],
    # Grupo 11 - Croacia y Adriático
    ["Dubrovnik", "Split", "Istria", "Hvar", "Montenegro Budva", "Albania Saranda", "Albania Ksamil"],
    # Grupo 12 - Turquía ampliada
    ["Antalya", "Bodrum", "Dalaman", "Fethiye", "Cesme", "Kusadasi", "Marmaris", "Alanya"],
    # Grupo 13 - Egipto ampliado
    ["Hurghada", "Sharm el Sheikh", "Marsa Alam", "El Gouna", "Luxor"],
    # Grupo 14 - Grecia continental
    ["Atenas", "Salónica", "Pelión", "Halkidiki", "Peloponeso"],
    # Grupo 15 - Europa otros seguros
    ["Malta", "Chipre", "Chipre Norte", "Eslovenia costa", "Bulgaria Sunny Beach"],
    # Grupo 16 - Portugal atlántico
    ["Algarve", "Madeira", "Azores", "Lisboa costa", "Porto"],
    # Grupo 17 - Temporadas y ofertas
    ["vacaciones verano 2025 oferta", "viaje invierno sol barato", "Semana Santa playa", "Navidad Caribe", "puente mayo destino", "última hora vacaciones playa"],
    # Grupo 18 - Comparativas
    ["Mallorca vs Creta", "Cancún vs Punta Cana", "Tenerife vs Gran Canaria", "Antalya vs Hurghada", "Maldivas vs Mauricio", "Costa del Sol vs Algarve", "Bali vs Tailandia"],
    # Grupo 19 - Temáticas
    ["vacaciones playa familia Europa", "mejor destino luna de miel", "todo incluido barato", "mejor isla griega parejas", "destino sostenible Europa", "viaje aventura seguro"],
    # Grupo 20 - Marcas TUI
    ["TUI Blue Mallorca", "TUI Magic Life Bodrum", "TUI Sensatori Tenerife", "Robinson Club Fuerteventura", "RIU Cancún", "Iberostar Creta", "Barceló Bávaro"],
    # Grupo 21 - Baleares
    ["Mallorca", "Ibiza", "Menorca", "Formentera"],
    # Grupo 22 - Canarias
    ["Tenerife", "Gran Canaria", "Lanzarote", "Fuerteventura", "La Palma", "La Gomera"],
    # Grupo 23 - Costas España
    ["Costa del Sol", "Costa Brava", "Costa Blanca", "Costa Dorada", "Almería", "Cádiz", "Huelva"],
    # Grupo 24 - Grecia islas
    ["Creta", "Santorini", "Rodas", "Kos", "Corfu", "Zante", "Mykonos", "Kefalonia", "Skiathos", "Paros", "Naxos"],
    # Grupo 25 - Caribe clásico
    ["Cancún", "Riviera Maya", "Punta Cana", "Cuba Varadero", "Jamaica Montego Bay", "Aruba", "Curaçao", "Barbados"],
    # Grupo 26 - Opiniones español
    ["Mallorca vacaciones opiniones", "Tenerife hotel todo incluido", "Cancún experiencia viaje", "Creta playa familiar", "Punta Cana resort opiniones", "Antalya all inclusive"],
    # Grupo 27 - Reviews inglés
    ["Mallorca holiday review", "Tenerife all inclusive", "Cancun resort review", "Crete family holiday", "Rhodes travel tips", "Maldives overwater villa review"],
    # Grupo 28 - Bewertungen alemán
    ["Mallorca Urlaub Erfahrung", "Teneriffa Hotel Bewertung", "Kreta Familienurlaub", "Antalya All Inclusive", "Hurghada Reise Tipps", "Fuerteventura Strand"]
]

RUTA_DESCARGAS = r"C:\Users\mtkyg\Downloads"
ARCHIVO_SALIDA = os.path.join(RUTA_DESCARGAS, "conectividad_y_pasajeros_2025.csv")

HUBS_EMISORES = {
    "España": ["MAD", "BCN", "AGP", "ALC"],
    "Reino Unido": ["LHR", "LGW", "STN", "MAN", "LTN"],
    "Alemania": ["FRA", "MUC", "BER", "DUS"]
}

# Códigos IATA de destinos de Largo Radio (requieren aviones de 300 plazas)
IATA_LARGO_RADIO = {
    "CUN", "PUJ", "MLE", "HKT", "DPS", "DXB", "AUH", "SJO", "PTY", "SJD", 
    "PVR", "BGI", "AUA", "CUR", "MBJ", "MRU", "SEZ", "ZNZ", "VRA", "RUN", 
    "MCT", "SLL", "TQO", "CMB", "DAD", "ANU", "UVF", "GND", "DOM", "BON", "PLS"
}

FACTOR_OCUPACION_MEDIO = 0.85  # 85% de ocupación media habitual

# ==============================================================================
# 1. CARGAR BASE DE DATOS DE RUTAS MUNDIALES (OPENFLIGHTS)
# ==============================================================================
print("Descargando base de datos mundial de rutas (OpenFlights)...")
URL_ROUTES = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat"
cols_routes = ["airline", "airline_id", "source_airport", "source_airport_id", "destination_airport", "destination_airport_id", "codeshare", "stops", "equipment"]
df_routes = pd.read_csv(URL_ROUTES, names=cols_routes, header=None, na_values="\\N")

db_airports_iata = airportsdata.load('IATA')

# ==============================================================================
# 2. DICCIONARIO DE MAPEO Y FUNCIONES AUXILIARES
# ==============================================================================
MAPEO_EXPLICITO = {
    "alicante": "ALC", "barcelona": "BCN", "benalmádena": "AGP", "benidorm": "ALC", "bilbao": "BIO",
    "cartagena": "ALC", "chiclana de la frontera": "XRY", "coruña": "LCG", "cádiz": "XRY", "córdoba": "SVQ",
    "san sebastian": "EAS", "eivissa": "IBZ", "elche": "ALC", "fuengirola": "AGP", "gijón": "OVD",
    "granada": "GRX", "jerez de la frontera": "XRY", "madrid": "MAD", "marbella": "AGP", "murcia": "RMU",
    "málaga": "AGP", "oviedo": "OVD", "palma": "PMI", "palmas de gran canaria": "LPA", "pamplona": "PNA",
    "puerto de santa maría": "XRY", "puerto de la cruz": "TFS", "salamanca": "SLM", "santa cruz de tenerife": "TFS",
    "santiago de compostela": "SCQ", "sevilla": "SVQ", "toledo": "MAD", "torremolinos": "AGP", "torrevieja": "ALC",
    "valladolid": "VLL", "valència": "VLC", "vigo": "VGO", "vitoria": "VIT", "zaragoza": "ZAZ",
    "batumi": "BUS", "portoroz": "POW", "lefkada": "PVK", "thassos": "KVA", "tropea": "SUF",
    "phuket": "HKT", "koh samui": "USM", "bali": "DPS", "da nang": "DAD", "sri lanka": "CMB", "maldivas": "MLE",
    "mauricio": "MRU", "seychelles": "SEZ", "reunión": "RUN", "zanzíbar": "ZNZ",
    "antigua": "ANU", "santa lucía": "UVF", "granada caribe": "GND", "dominica": "DOM", "bonaire": "BON", "turks and caicos": "PLS",
    "islandia": "KEF", "noruega": "OSL", "azores": "PDL", "madeira": "FNC", "canarias": "TFS", "escocia": "EDI",
    "dubái": "DXB", "abu dhabi": "AUH", "muscat": "MCT", "salalah": "SLL", "jordania": "AMM", "aqaba": "AQJ",
    "playa del carmen": "CUN", "tulum": "TQO", "los cabos": "SJD", "puerto vallarta": "PVR", "costa rica": "SJO", "panamá": "PTY",
    "sal": "SID", "boa vista": "BVC", "santiago": "RAI", "santo antão": "VXE",
    "marrakech": "RAK", "agadir": "AGA", "essaouira": "ESU", "fez": "FEZ", "tánger": "TNG",
    "sicilia": "CTA", "cerdeña": "CAG", "calabria": "SUF", "puglia": "BRI", "costa amalfitana": "NAP", "toscana": "FLR",
    "dubrovnik": "DBV", "split": "SPU", "istria": "PUY", "hvar": "SPU", "budva": "TIV", "saranda": "TIA", "ksamil": "CFU",
    "antalya": "AYT", "bodrum": "BJV", "dalaman": "DLM", "fethiye": "DLM", "cesme": "ADB", "kusadasi": "ADB", "marmaris": "DLM", "alanya": "GZP",
    "hurghada": "HRG", "sharm el sheikh": "SSH", "marsa alam": "RMF", "el gouna": "HRG", "luxor": "LXR",
    "atenas": "ATH", "salónica": "SKG", "peloponeso": "GPA",
    "malta": "MLA", "chipre": "LCA", "sunny beach": "BOJ", "albania": "TIA",
    "algarve": "FAO", "porto": "OPO", "lisboa": "LIS",
    "mallorca": "PMI", "ibiza": "IBZ", "menorca": "MAH", "formentera": "IBZ",
    "tenerife": "TFS", "gran canaria": "LPA", "lanzarote": "ACE", "fuerteventura": "FUE", "la palma": "SPC", "la gomera": "GMZ",
    "costa del sol": "AGP", "costa brava": "GRO", "costa blanca": "ALC", "costa dorada": "REU", "almería": "LEI", "huelva": "FAO",
    "creta": "HER", "santorini": "JTR", "rodas": "RHO", "kos": "KGS", "corfu": "CFU", "zante": "ZTH", "mykonos": "JMK",
    "kefalonia": "EFL", "skiathos": "JSI", "paros": "PAS", "naxos": "JNX",
    "cancún": "CUN", "cancun": "CUN", "riviera maya": "CUN", "punta cana": "PUJ", "varadero": "VRA", "montego bay": "MBJ", "aruba": "AUA", "curaçao": "CUR", "barbados": "BGI"
}

def resolver_iata(cadena):
    cadena_clean = str(cadena).lower()
    for k, iata in MAPEO_EXPLICITO.items():
        if k in cadena_clean:
            return iata
            
    for iata, datos in db_airports_iata.items():
        city = datos.get("city", "").lower()
        if city and len(city) > 3 and city in cadena_clean:
            return iata
    return None

def obtener_asientos_por_vuelo(iata):
    """Devuelve 300 asientos para largo radio y 180 para corto/medio radio."""
    if iata in IATA_LARGO_RADIO:
        return 300
    return 180

# ==============================================================================
# 3. PROCESAMIENTO Y CÁLCULO DE PASAJEROS
# ==============================================================================
todos_los_elementos = []

for ciudad in DESTINOS_CIUDADES_ESPANOLAS:
    todos_los_elementos.append({"origen_grupo": "Ciudades Españolas", "termino": ciudad})

for idx, grupo in enumerate(DESTINOS_GRUPOS, 1):
    nombre_grupo = f"Grupo {idx}"
    for termino in grupo:
        todos_los_elementos.append({"origen_grupo": nombre_grupo, "termino": termino})

print(f"\nProcesando {len(todos_los_elementos)} elementos totales...")

resultados = []

for item in todos_los_elementos:
    termino = item["termino"]
    grupo = item["origen_grupo"]
    iata_dest = resolver_iata(termino)
    
    if not iata_dest:
        resultados.append({
            "grupo": grupo,
            "termino_original": termino,
            "iata_destino": "Sin aeropuerto directo",
            "rutas_directas_ES": 0,
            "rutas_directas_UK": 0,
            "rutas_directas_DE": 0,
            "vuelos_semanales_estimados": 0,
            "asientos_semanales_ofertados": 0,
            "pasajeros_semanales_estimados": 0,      # <-- NUEVA VARIABLE CLAVE
            "pasajeros_anuales_estimados": 0
        })
        continue

    # Filtrar rutas de OpenFlights hacia este aeropuerto
    rutas_hacia_destino = df_routes[df_routes["destination_airport"] == iata_dest]
    
    rutas_es = len(rutas_hacia_destino[rutas_hacia_destino["source_airport"].isin(HUBS_EMISORES["España"])])
    rutas_uk = len(rutas_hacia_destino[rutas_hacia_destino["source_airport"].isin(HUBS_EMISORES["Reino Unido"])])
    rutas_de = len(rutas_hacia_destino[rutas_hacia_destino["source_airport"].isin(HUBS_EMISORES["Alemania"])])
    
    # 1. Estimación de frecuencia semanal de vuelos
    frec_semanal_vuelos = (rutas_es * 7) + (rutas_uk * 10) + (rutas_de * 9)
    
    # 2. Estimación de plazas / asientos
    capacidad_avión = obtener_asientos_por_vuelo(iata_dest)
    asientos_semanales = frec_semanal_vuelos * capacidad_avión
    
    # 3. Estimación de Pasajeros Reales (aplicando Ocupación del 85%)
    pax_semanales = round(asientos_semanales * FACTOR_OCUPACION_MEDIO)
    pax_anuales = round(pax_semanales * 52)

    resultados.append({
        "grupo": grupo,
        "termino_original": termino,
        "iata_destino": iata_dest,
        "rutas_directas_ES": rutas_es,
        "rutas_directas_UK": rutas_uk,
        "rutas_directas_DE": rutas_de,
        "vuelos_semanales_estimados": frec_semanal_vuelos,
        "asientos_semanales_ofertados": asientos_semanales,
        "pasajeros_semanales_estimados": pax_semanales,   # <-- CAPACIDAD REAL SEMANAL
        "pasajeros_anuales_estimados": pax_anuales
    })

# ==============================================================================
# 4. EXPORTAR A CSV
# ==============================================================================
df_final = pd.DataFrame(resultados)
df_final.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8-sig")

print(f"\n¡Proceso completado!")
print(f"Dataset guardado en: {ARCHIVO_SALIDA}")

Descargando base de datos mundial de rutas (OpenFlights)...

Procesando 212 elementos totales...

¡Proceso completado!
Dataset guardado en: C:\Users\mtkyg\Downloads\conectividad_y_pasajeros_2025.csv


## 4. Seguridad, Infraestructura y Salud
*Indicadores de fricción y factores de decisión en la fase de consideración.*

* **Índice de Seguridad / Tasa de Criminalidad Percibida:** Evaluación de la percepción de seguridad en el destino.
* **Calidad de la Infraestructura Sanitaria:** Accesibilidad a centros médicos y cobertura hospitalaria para turistas.
* **Requisitos Burocráticos y Visados:** 
  * Pertenencia a espacio Schengen.
  * Exención de visado / Visado a la llegada (*Visa on Arrival*) / Trámites previos (e-Visa).



In [ ]:
#Datos de homicidios cada 100mil habitantes por pais y datos de sanidad.

import requests
import pandas as pd

def obtener_indicador_banco_mundial(indicator_code, column_name):
    """
    Función genérica para consultar cualquier indicador de la API del Banco Mundial.
    """
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}?date=2015:2024&format=json&per_page=2000"
    headers = {"User-Agent": "TesisAcademicaTurismoBot/1.0"}
    
    try:
        response = requests.get(url, headers=headers, timeout=25)
        if response.status_code == 200:
            data = response.json()
            if len(data) > 1 and data[1]:
                registros = []
                for item in data[1]:
                    pais = item.get("country", {}).get("value")
                    iso = item.get("countryiso3code")
                    año = item.get("date")
                    valor = item.get("value")
                    
                    if valor is not None and iso:
                        registros.append({
                            "iso": iso,
                            "pais": pais,
                            "año": int(año),
                            column_name: float(valor)
                        })
                return pd.DataFrame(registros)
    except Exception as e:
        print(f"Error al obtener el indicador {indicator_code}: {e}")
        
    return pd.DataFrame()


print("Descargando datos de infraestructura sanitaria (Camas de hospital)...")
df_camas = obtener_indicador_banco_mundial("SH.MED.BEDS.ZS", "camas_hospital_1000hab")

print("Descargando datos de seguridad ciudadana (Tasa de homicidios)...")
df_homicidios = obtener_indicador_banco_mundial("VC.IHR.PSRC.P5", "tasa_homicidios_100mil")


# Procesar y filtrar el registro más reciente para cada país
if not df_camas.empty:
    df_camas_reciente = df_camas.sort_values("año", ascending=False).drop_duplicates(subset=["iso"]).drop(columns=["año"])
else:
    df_camas_reciente = pd.DataFrame(columns=["iso", "pais", "camas_hospital_1000hab"])

if not df_homicidios.empty:
    df_homicidios_reciente = df_homicidios.sort_values("año", ascending=False).drop_duplicates(subset=["iso"]).drop(columns=["año"])
else:
    df_homicidios_reciente = pd.DataFrame(columns=["iso", "pais", "tasa_homicidios_100mil"])


# Fusionar ambos datasets mediante un Outer Join por código ISO y País
if not df_camas_reciente.empty and not df_homicidios_reciente.empty:
    df_seguridad_sanidad_unificado = pd.merge(
        df_camas_reciente, 
        df_homicidios_reciente, 
        on=["iso", "pais"], 
        how="outer"
    )
    
    print("\n--- MATRICES UNIFICADAS: SANIDAD Y SEGURIDAD (BANCO MUNDIAL) ---")
    print(df_seguridad_sanidad_unificado.dropna(subset=["camas_hospital_1000hab", "tasa_homicidios_100mil"]).head(15).to_string(index=False))
else:
    print("No se pudieron fusionar los datos correctamente.")

Descargando datos de infraestructura sanitaria (Camas de hospital)...
Descargando datos de seguridad ciudadana (Tasa de homicidios)...

--- MATRICES UNIFICADAS: SANIDAD Y SEGURIDAD (BANCO MUNDIAL) ---
iso                pais  camas_hospital_1000hab  tasa_homicidios_100mil
AFG         Afghanistan                0.350000                4.032458
AGO              Angola                0.750000                4.098267
ALB             Albania                2.900000                1.387083
ARB          Arab World                1.214107                3.800000
ARG           Argentina                3.360000                4.492911
ARM             Armenia                4.240000                2.208336
ATG Antigua and Barbuda                3.420000               10.716333
AUS           Australia                3.820000                0.854406
AUT             Austria                6.700000                0.876191
AZE          Azerbaijan                3.680000                2.161228
BEL    

In [22]:
TODOS_LOS_DESTINOS = [
    # Ciudades Españolas
    'Alicante', 'Barcelona', 'Benalmádena', 'Benidorm', 'Bilbao', 'Cartagena', 
    'Chiclana de la Frontera', 'Coruña', 'Cádiz', 'Córdoba', 'San Sebastian', 
    'Eivissa', 'Elche', 'Fuengirola', 'Gijón', 'Granada', 'Jerez de la Frontera', 
    'Madrid', 'Marbella', 'Murcia', 'Málaga', 'Oviedo', 'Palma', 'Palmas de Gran Canaria', 
    'Pamplona', 'Puerto de Santa María', 'Puerto de la Cruz', 'Salamanca', 
    'Santa Cruz de Tenerife', 'Santiago de Compostela', 'Sevilla', 'Toledo', 
    'Torremolinos', 'Torrevieja', 'Valladolid', 'València', 'Vigo', 'Vitoria', 'Zaragoza',
    
    # Grupos y Destinos
    'Georgia Batumi', 'Eslovenia Portoroz', 'Croacia islas', 'Grecia Lefkada', 'Grecia Thassos', 'Italia Tropea',
    'Tailandia Phuket', 'Tailandia Koh Samui', 'Bali', 'Vietnam Da Nang', 'Sri Lanka', 'Maldivas',
    'Mauricio', 'Seychelles', 'Reunión', 'Zanzíbar', 'Maldivas atolón',
    'Antigua', 'Santa Lucía', 'Granada Caribe', 'Dominica', 'Bonaire', 'Turks and Caicos',
    'Islandia', 'Noruega fiordos', 'Azores senderismo', 'Madeira trekking', 'Canarias volcanes', 'Escocia Highlands',
    'Dubái', 'Abu Dhabi', 'Omán Muscat', 'Omán Salalah', 'Jordania Mar Muerto', 'Jordania Aqaba',
    'Playa del Carmen', 'Tulum', 'Los Cabos', 'Puerto Vallarta', 'Costa Rica', 'Panamá',
    'Cabo Verde Sal', 'Cabo Verde Boa Vista', 'Cabo Verde Santiago', 'Cabo Verde Santo Antão',
    'Marrakech', 'Agadir', 'Essaouira', 'Fez', 'Tánger',
    'Sicilia', 'Cerdeña', 'Calabria', 'Puglia', 'Costa Amalfitana', 'Toscana', 'Cinque Terre', 'Lago de Garda',
    'Dubrovnik', 'Split', 'Istria', 'Hvar', 'Montenegro Budva', 'Albania Saranda', 'Albania Ksamil',
    'Antalya', 'Bodrum', 'Dalaman', 'Fethiye', 'Cesme', 'Kusadasi', 'Marmaris', 'Alanya',
    'Hurghada', 'Sharm el Sheikh', 'Marsa Alam', 'El Gouna', 'Luxor',
    'Atenas', 'Salónica', 'Pelión', 'Halkidiki', 'Peloponeso',
    'Malta', 'Chipre', 'Chipre Norte', 'Eslovenia costa', 'Bulgaria Sunny Beach',
    'Algarve', 'Madeira', 'Azores', 'Lisboa costa', 'Porto',
    'Mallorca', 'Ibiza', 'Menorca', 'Formentera',
    'Tenerife', 'Gran Canaria', 'Lanzarote', 'Fuerteventura', 'La Palma', 'La Gomera', 'Almería', 'Huelva',
    'Creta', 'Santorini', 'Rodas', 'Kos', 'Corfu', 'Zante', 'Mykonos', 'Kefalonia', 'Skiathos', 'Paros', 'Naxos',
    'Cancún', 'Riviera Maya', 'Punta Cana', 'Cuba Varadero', 'Jamaica Montego Bay', 'Aruba', 'Curaçao', 'Barbados'
    ]



In [3]:
#Devuelve niveles de seguridad del 1-4 para cada 195 paies.

def obtener_seguridad_pais(pais_iso2):
    """Consulta el nivel de advertencia de viaje (1-4) para un país."""
    url = f"https://www.travel-advisory.info/api?countrycode={pais_iso2}"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    # La respuesta viene anidada por código de país
    pais_data = list(data["data"].values())[0] if data.get("data") else None
    if pais_data:
        return pais_data["advisory"]["score"]  # score numérico de riesgo
    return None

agregamos el pais al que pertenece cada ciudad

## 6. Atractivos y Tipología del Destino (Categorización y Clustering)
*Atributos estructurales utilizados para la recomendación por similitud entre destinos.*

* **Kilómetros de Playa y Banderas Azules:** Extensión de costa y sellos de calidad ambiental.
* **Patrimonio Cultural / UNESCO:** Densidad de sitios declarados Patrimonio de la Humanidad o puntos de interés cultural.
* **Perfil de Oferta Complementaria:** 
  * Densidad de oferta gastronómica (locales tradicionales vs. alta cocina).
  * Opciones de ocio nocturno vs. turismo activo / naturaleza.

In [29]:
import requests
import pandas as pd

def obtener_unesco_wikidata():
    """
    Consulta la API SPARQL de Wikidata para obtener el recuento real 
    de bienes declarados Patrimonio de la Humanidad por la UNESCO por país.
    """
    url = "https://query.wikidata.org/sparql"
    
    # Consulta SPARQL para extraer países y número de sitios UNESCO vinculados
    query = """
    SELECT ?countryLabel (COUNT(?site) as ?count) WHERE {
      ?site wdt:P1435 wd:Q9259. # Q9259 = Patrimonio de la Humanidad
      ?site wdt:P17 ?country.
      SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
    } GROUP BY ?countryLabel ORDER BY DESC(?count) LIMIT 15
    """
    
    headers = {
        "User-Agent": "TesisAcademicaTurismoBot/1.0 (investigacion@uam.es)",
        "Accept": "application/json"
    }
    
    print("Consultando la API de Wikidata (UNESCO)...")
    try:
        response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=20)
        if response.status_code == 200:
            data = response.json()["results"]["bindings"]
            registros = []
            for item in data:
                registros.append({
                    "pais": item["countryLabel"]["value"],
                    "sitios_unesco": int(item["count"]["value"])
                })
            return pd.DataFrame(registros)
        else:
            print(f"Error en API Wikidata: {response.status_code}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Excepción en Wikidata: {e}")
        return pd.DataFrame()


def consultar_oferta_osm(ciudad="Ibiza"):
    """
    Consulta la API Overpass de OpenStreetMap (OSM) en tiempo real para 
    contar la infraestructura real de restauración y ocio nocturno de un destino.
    """
    url = "https://overpass-api.de/api/interpreter"
    
    # Consulta Overpass QL buscando elementos dentro del límite administrativo de la ciudad
    query = f"""
    [out:json][timeout:25];
    area[name="{ciudad}"]->.searchArea;
    (
      node["amenity"~"restaurant|bar|nightclub|pub"](area.searchArea);
      way["amenity"~"restaurant|bar|nightclub|pub"](area.searchArea);
    );
    out tags;
    """
    
    headers = {"User-Agent": "TesisTuristicamadrid/1.0"}
    
    try:
        response = requests.post(url, data={"data": query}, headers=headers, timeout=25)
        if response.status_code == 200:
            elementos = response.json().get("elements", [])
            
            restaurantes = 0
            ocio_nocturno = 0
            
            for el in elementos:
                amenity = el.get("tags", {}).get("amenity", "")
                if amenity == "restaurant":
                    restaurantes += 1
                elif amenity in ["bar", "nightclub", "pub"]:
                    ocio_nocturno += 1
                    
            return {
                "destino": ciudad,
                "restaurantes_osm": restaurantes,
                "ocio_nocturno_osm": ocio_nocturno
            }
        else:
            print(f"Error en OpenStreetMap para '{ciudad}': Código HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"Excepción al conectar con OpenStreetMap para '{ciudad}': {e}")
        return None


# --- EJECUCIÓN DE LAS APIS ---

# 1. Obtener datos reales de patrimonio cultural (UNESCO)
df_unesco_real = obtener_unesco_wikidata()
if not df_unesco_real.empty:
    print("\n--- DATOS REALES: PATRIMONIO UNESCO POR PAÍS ---")
    print(df_unesco_real.head(10).to_string(index=False))

# 2. Obtener datos reales de oferta complementaria (OpenStreetMap) para destinos clave
destinos_analizar = ["Madrid", "Ibiza", "Atenas"]
resultados_osm = []

print("\nConsultando infraestructura turística en OpenStreetMap API...")
for dest in destinos_analizar:
    res = consultar_oferta_osm(dest)
    if res:
        resultados_osm.append(res)

if resultados_osm:
    df_osm_real = pd.DataFrame(resultados_osm)
    print("\n--- DATOS REALES: OFERTA COMPLEMENTARIA (OSM API) ---")
    print(df_osm_real.to_string(index=False))

Consultando la API de Wikidata (UNESCO)...

--- DATOS REALES: PATRIMONIO UNESCO POR PAÍS ---
                   pais  sitios_unesco
                 Brasil           1471
                Francia            201
               Alemania            107
                 Italia            105
                 España             93
República Popular China             87
                   Perú             73
                Bélgica             62
                  India             56
                 México             50

Consultando infraestructura turística en OpenStreetMap API...
Error en OpenStreetMap para 'Madrid': Código HTTP 504

--- DATOS REALES: OFERTA COMPLEMENTARIA (OSM API) ---
destino  restaurantes_osm  ocio_nocturno_osm
  Ibiza                74                 15
 Atenas                47                 25


In [23]:
import time
import requests
import pandas as pd


def obtener_unesco_wikidata():
    """
    Consulta la API SPARQL de Wikidata para obtener el recuento real 
    de bienes declarados Patrimonio de la Humanidad por la UNESCO por país.
    """
    url = "https://query.wikidata.org/sparql"
    query = """
    SELECT ?countryLabel (COUNT(?site) as ?count) WHERE {
      ?site wdt:P1435 wd:Q9259. 
      ?site wdt:P17 ?country.
      SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
    } GROUP BY ?countryLabel ORDER BY DESC(?count)
    """
    headers = {"User-Agent": "TesisAcademicaTurismoBot/1.0", "Accept": "application/json"}
    
    try:
        response = requests.get(url, params={"query": query, "format": "json"}, headers=headers, timeout=25)
        if response.status_code == 200:
            data = response.json()["results"]["bindings"]
            diccionario_unesco = {}
            for item in data:
                pais = item["countryLabel"]["value"].lower()
                count = int(item["count"]["value"])
                diccionario_unesco[pais] = count
            return diccionario_unesco
    except Exception as e:
        print(f"Excepción en Wikidata: {e}")
    return {}


def consultar_oferta_osm(destino):
    """
    Consulta la API Overpass de OpenStreetMap (OSM) para contar la infraestructura 
    real de restaurantes y ocio nocturno para un destino de la lista.
    """
    url = "https://overpass-api.de/api/interpreter"
    
    # Limpiamos el nombre si tiene varias palabras (ej: "Georgia Batumi" -> "Batumi")
    nombre_ciudad = destino.split()[-1]

    query = f"""
    [out:json][timeout:25];
    area[name="{nombre_ciudad}"]->.searchArea;
    (
      node["amenity"~"restaurant|bar|nightclub|pub"](area.searchArea);
      way["amenity"~"restaurant|bar|nightclub|pub"](area.searchArea);
    );
    out tags;
    """
    
    headers = {"User-Agent": "TesisTuristicamadrid/1.0"}
    
    try:
        response = requests.post(url, data={"data": query}, headers=headers, timeout=30)
        if response.status_code == 200:
            elementos = response.json().get("elements", [])
            restaurantes = 0
            ocio_nocturno = 0
            
            for el in elementos:
                amenity = el.get("tags", {}).get("amenity", "")
                if amenity == "restaurant":
                    restaurantes += 1
                elif amenity in ["bar", "nightclub", "pub"]:
                    ocio_nocturno += 1
                    
            return {
                "destino": destino,
                "restaurantes_osm": restaurantes,
                "ocio_nocturno_osm": ocio_nocturno
            }
    except Exception as e:
        pass
        
    return {
        "destino": destino,
        "restaurantes_osm": 0,
        "ocio_nocturno_osm": 0
    }


# --- PROCESAMIENTO MASIVO DE TODOS LOS DESTINOS ---

print("1. Descargando datos de UNESCO desde Wikidata...")
mapa_unesco = obtener_unesco_wikidata()

print(f"\n2. Consultando OpenStreetMap (OSM) para los {len(TODOS_LOS_DESTINOS)} destinos...")
print("Nota: Este proceso puede tardar unos minutos debido a las pausas de cortesía de la API.\n")

resultados_totales = []

for i, dest in enumerate(TODOS_LOS_DESTINOS):
    # Consultar oferta OSM
    datos_osm = consultar_oferta_osm(dest)
    
    # Intentar asociar UNESCO según el país deducido del destino
    # (Ejemplo básico de asignación por coincidencia de texto)
    sitios_unesco = 0
    for pais_db, count in mapa_unesco.items():
        if pais_db in dest.lower() or (dest.lower() in ["madrid", "barcelona", "valència", "sevilla", "granada", "ibiza", "mallorca", "tenerife"] and pais_db == "españa"):
            sitios_unesco = count
            break
            
    datos_osm["sitios_unesco_pais"] = sitios_unesco
    resultados_totales.append(datos_osm)
    
    # Imprimir progreso cada 10 destinos
    if (i + 1) % 10 == 0:
        print(f"Procesados {i + 1} de {len(TODOS_LOS_DESTINOS)} destinos...")
        
    time.sleep(1.2) # Pausa obligatoria para no saturar los servidores de OpenStreetMap

# Generar DataFrame final unificado
df_final_destinos = pd.DataFrame(resultados_totales)

print("\n--- MATRICES FINALES DE ATRIBUTOS PARA TU LISTA ---")
print(df_final_destinos.head(15).to_string(index=False))

1. Descargando datos de UNESCO desde Wikidata...

2. Consultando OpenStreetMap (OSM) para los 163 destinos...
Nota: Este proceso puede tardar unos minutos debido a las pausas de cortesía de la API.

Procesados 10 de 163 destinos...
Procesados 20 de 163 destinos...
Procesados 30 de 163 destinos...
Procesados 40 de 163 destinos...


KeyboardInterrupt: 

In [7]:
import requests
import time
import pandas as pd

def geocodificar(destino):
    """Resuelve un nombre de destino a sus coordenadas y país vía Nominatim (OSM)."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": destino, "format": "json", "limit": 1, "addressdetails": 1}
    headers = {"User-Agent": "TesisAcademicaTurismoBot/1.0 (contacto@ejemplo.com)"}

    resp = requests.get(url, params=params, headers=headers, timeout=15)
    resp.raise_for_status()
    resultados = resp.json()

    if not resultados:
        return None

    r = resultados[0]
    return {
        "lat": float(r["lat"]),
        "lon": float(r["lon"]),
        "pais": r.get("address", {}).get("country", None),
        "display_name": r.get("display_name"),
    }

In [8]:
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
]

In [ ]:
def consultar_overpass_por_coordenadas(lat, lon, radio_metros=15000):
    query = f"""
    [out:json][timeout:25];
    (
      node["historic"](around:{radio_metros},{lat},{lon});
      way["historic"](around:{radio_metros},{lat},{lon});
      node["tourism"="museum"](around:{radio_metros},{lat},{lon});
      way["tourism"="museum"](around:{radio_metros},{lat},{lon});
    )->.patrimonio;
    (
      node["natural"="beach"](around:{radio_metros},{lat},{lon});
      way["natural"="beach"](around:{radio_metros},{lat},{lon});
      node["leisure"="beach_resort"](around:{radio_metros},{lat},{lon});
    )->.naturaleza;
    .patrimonio out count;
    .naturaleza out count;
    """
    headers = {"User-Agent": "TesisAcademicaTurismoBot/1.0 (contacto@ejemplo.com)"}

    ultimo_error = None
    for url in OVERPASS_URLS:
        try:
            resp = requests.post(url, data={"data": query}, headers=headers, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            elementos = data.get("elements", [])
            patrimonio = int(elementos[0].get("tags", {}).get("total", 0)) if len(elementos) > 0 else 0
            naturaleza = int(elementos[1].get("tags", {}).get("total", 0)) if len(elementos) > 1 else 0
            return patrimonio, naturaleza
        except Exception as e:
            print(f"    Mirror {url} falló: {e}")
            ultimo_error = e
            time.sleep(2)  # pequeña pausa antes de probar el siguiente mirror

    raise ultimo_error  # si los 3 mirrors fallaron, recién ahí propagamos el error

In [10]:
def obtener_seguridad_pais(pais_iso2):
    """Consulta el score de riesgo de viaje (0-5, más alto = más riesgo) para un país."""
    url = f"https://www.travel-advisory.info/api?countrycode={pais_iso2}"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    pais_data = list(data["data"].values())[0] if data.get("data") else None
    return pais_data["advisory"]["score"] if pais_data else None

# Prueba con un par de países conocidos
for codigo in ["ES", "GR", "TZ"]:  # España, Grecia, Tanzania
    score = obtener_seguridad_pais(codigo)
    print(f"{codigo}: score de riesgo = {score}")

SSLError: HTTPSConnectionPool(host='www.travel-advisory.info', port=443): Max retries exceeded with url: /api?countrycode=ES (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'www.travel-advisory.info'. (_ssl.c:1081)")))

In [49]:
def procesar_destino(destino):
    try:
        geo = geocodificar(destino)
        if geo is None:
            return {"destino": destino, "pais": None, "patrimonio_cultural_historico": None, "naturaleza_playa": None, "error": "No geocodificado"}

        time.sleep(1)  # respetamos el límite de Nominatim antes de la siguiente llamada (Overpass)

        patrimonio, naturaleza = consultar_overpass_por_coordenadas(geo["lat"], geo["lon"])

        return {
            "destino": destino,
            "pais": geo["pais"],
            "patrimonio_cultural_historico": patrimonio,
            "naturaleza_playa": naturaleza,
            "error": None,
        }
    except Exception as e:
        return {"destino": destino, "pais": None, "patrimonio_cultural_historico": None, "naturaleza_playa": None, "error": str(e)}
        

In [46]:
def peticion_con_reintentos(func, *args, max_intentos=4, **kwargs):
    """
    Ejecuta una función que hace una petición HTTP, reintentando con backoff
    creciente si falla por rate limit (429) o timeout del servidor (504/502/503).
    """
    for intento in range(1, max_intentos + 1):
        try:
            return func(*args, **kwargs)
        except requests.exceptions.HTTPError as e:
            status = e.response.status_code if e.response is not None else None
            if status in (429, 502, 503, 504):
                espera = 3 * intento  # 3s, 6s, 9s, 12s
                print(f"    Intento {intento}: error {status}. Esperando {espera}s...")
                time.sleep(espera)
            else:
                raise  # otros errores HTTP los dejamos explotar, no son transitorios
        except requests.exceptions.Timeout:
            espera = 3 * intento
            print(f"    Intento {intento}: timeout de conexión. Esperando {espera}s...")
            time.sleep(espera)
    raise Exception(f"Falló tras {max_intentos} intentos")

In [47]:
def procesar_destino(destino):
    try:
        geo = peticion_con_reintentos(geocodificar, destino)
        if geo is None:
            return {"destino": destino, "pais": None, "patrimonio_cultural_historico": None, "naturaleza_playa": None, "error": "No geocodificado"}

        time.sleep(1.5)  # margen extra sobre el límite de 1 req/seg de Nominatim

        patrimonio, naturaleza = peticion_con_reintentos(consultar_overpass_por_coordenadas, geo["lat"], geo["lon"])

        return {
            "destino": destino,
            "pais": geo["pais"],
            "patrimonio_cultural_historico": patrimonio,
            "naturaleza_playa": naturaleza,
            "error": None,
        }
    except Exception as e:
        return {"destino": destino, "pais": None, "patrimonio_cultural_historico": None, "naturaleza_playa": None, "error": str(e)}

In [51]:
resultados_completos = []
total = len(TODOS_LOS_DESTINOS)

for i, dest in enumerate(TODOS_LOS_DESTINOS, 1):
    print(f"[{i}/{total}] Procesando: {dest}...")
    res = procesar_destino(dest)
    resultados_completos.append(res)
    time.sleep(1.5)

df_patrimonio_naturaleza = pd.DataFrame(resultados_completos)

# Guardamos a CSV inmediatamente, para no perder el trabajo si algo corta la sesión después
df_patrimonio_naturaleza.to_csv("patrimonio_naturaleza_destinos.csv", index=False)

print(f"\nCompletado: {len(df_patrimonio_naturaleza)} destinos procesados")
print(f"Con error: {df_patrimonio_naturaleza['error'].notna().sum()}")

[1/163] Procesando: Alicante...
[2/163] Procesando: Barcelona...
[3/163] Procesando: Benalmádena...
    Mirror https://overpass-api.de/api/interpreter falló: 429 Client Error: Too Many Requests for url: https://overpass-api.de/api/interpreter
    Mirror https://overpass.kumi.systems/api/interpreter falló: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=30)
    Mirror https://lz4.overpass-api.de/api/interpreter falló: 504 Server Error: Gateway Timeout for url: https://lz4.overpass-api.de/api/interpreter
[4/163] Procesando: Benidorm...
[5/163] Procesando: Bilbao...
    Mirror https://overpass-api.de/api/interpreter falló: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
    Mirror https://overpass.kumi.systems/api/interpreter falló: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=30)
[6/163] Procesando: Cartagena...
[7/163] Procesando: Chiclana de la Frontera...
[8/163]